In [9]:
import pandas as pd

In [10]:
df = pd.read_csv('../data/cps_00012.csv')

# drop rows with OCC = 0 or na
df = df[(df['OCC'] != 0) & (df['OCC'].notna())]

In [11]:
print(df['OCC'].nunique())

526


In [19]:
print(df.columns)

Index(['YEAR', 'SERIAL', 'MONTH', 'HWTFINL', 'CPSID', 'ASECFLAG', 'PERNUM',
       'WTFINL', 'CPSIDP', 'CPSIDV', 'AGE', 'OCC', 'IND', 'OCC_str'],
      dtype='str')


In [12]:
soc_crosswalk = pd.read_excel('../data/2018-occupation-code-list-and-crosswalk.xlsx', sheet_name="2018 Census Occ Code List", skiprows=4, usecols="B:D")

In [13]:
print(df.shape[0])

2540430


In [14]:
# convert both columns to same type before merge
df['OCC_str'] = df['OCC'].astype(str).str.zfill(4)
soc_crosswalk['code_str'] = soc_crosswalk['2018 Census Code'].astype(str).str.strip()

df['OCC_str'].dtype, soc_crosswalk['code_str'].dtype

(<StringDtype(storage='python', na_value=nan)>,
 <StringDtype(storage='python', na_value=nan)>)

In [15]:
merged = df.merge(soc_crosswalk, how='left', indicator=True, left_on='OCC_str', right_on='code_str')

In [16]:
# check which codes did not merge
merged['_merge'].value_counts()

_merge
both          2540142
left_only         288
right_only          0
Name: count, dtype: int64

In [17]:
# only OCC code 9840 "Military Occupations" did not merge
didnt_merge = merged[merged['_merge'] == 'left_only']
print(didnt_merge['OCC_str'].unique())

<StringArray>
['9840']
Length: 1, dtype: str


In [18]:
print(merged['OCC'].nunique())
print(merged['code_str'].nunique())

526
525


In [18]:
# create quarters
print(merged['MONTH'].dtype)
merged['Quarter'] = (merged['MONTH'] - 1) // 3 + 1
print(merged['Quarter'].unique())

# create periods
merged['Period'] = merged['YEAR'].astype(str) + 'Q' + merged['Quarter'].astype(str)
print(merged['Period'].unique())

int64
[1 2 3 4]
<StringArray>
['2022Q1', '2022Q2', '2022Q3', '2022Q4', '2023Q1', '2023Q2', '2023Q3',
 '2023Q4', '2024Q1', '2024Q2', '2024Q3', '2024Q4', '2025Q1', '2025Q2',
 '2025Q3', '2025Q4', '2026Q1', '2026Q2']
Length: 18, dtype: str


In [25]:
cell_counts = merged.groupby(['OCC_str', 'Period']).size().unstack(fill_value=0)
x = 30
valid = cell_counts >= x

valid.sum(axis=0)

groups_all_periods = valid.all(axis=1)

groups_all_periods[groups_all_periods].index

Index(['0010', '0020', '0051', '0052', '0060', '0102', '0110', '0120', '0136',
       '0137',
       ...
       '9150', '9365', '9510', '9600', '9610', '9620', '9640', '9645', '9720',
       '9760'],
      dtype='str', name='OCC_str', length=336)